# Reducing `raw_data_household_power_consumption.txt` for Analysis

This notebook shows the exact code used to reduce the raw household power dataset into a smaller dataset suitable for analysis.

In [2]:
import pandas as pd
from pathlib import Path

# Source file
source = Path(r"C:\Users\amissetu\Documents\GitHub\final_project\wattwise-energy-analytics\raw_data_household_power_consumption.txt")
output = Path(r"C:\Users\amissetu\Documents\GitHub\final_project\wattwise-energy-analytics\reduced_data")

# 1) Load the raw dataset
df = pd.read_csv(
    source,
    sep=';',
    na_values='?',
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True,
    low_memory=False
)

# 2) Select the main energy columns
metric_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

# 3) Convert all metric columns to numeric values
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4) Set datetime as the index and sort chronologically
df = df[['datetime', *metric_cols]].sort_values('datetime').set_index('datetime')

# 5) Aggregate minute-level data into hourly averages
hourly_df = df.resample('H').mean().reset_index()

# 6) Split datetime into date and time columns for readability
hourly_df['date'] = hourly_df['datetime'].dt.strftime('%Y-%m-%d')
hourly_df['time'] = hourly_df['datetime'].dt.strftime('%H:%M:%S')

# 7) Keep only the final analysis-ready columns
final_df = hourly_df[['date', 'time', *metric_cols]].copy()

# 8) Save the reduced dataset
final_df.to_csv(output, index=False, float_format='%.4f')

print(f'Original rows: {len(df)}')
print(f'Reduced rows: {len(final_df)}')
print(final_df.head().to_string(index=False))


Original rows: 2075259
Reduced rows: 34589
      date     time  Global_active_power  Global_reactive_power    Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3
2006-12-16 17:00:00             4.222889               0.229000 234.643889         18.100000             0.0        0.527778       16.861111
2006-12-16 18:00:00             3.632200               0.080033 234.580167         15.600000             0.0        6.716667       16.866667
2006-12-16 19:00:00             3.400233               0.085233 233.232500         14.503333             0.0        1.433333       16.683333
2006-12-16 20:00:00             3.268567               0.075100 234.071500         13.916667             0.0        0.000000       16.783333
2006-12-16 21:00:00             3.056467               0.076667 237.158667         13.046667             0.0        0.416667       17.216667


## Step-by-step explanation

1. Loaded the raw data file using `pandas.read_csv()` and specified `sep=';'` because the raw file is semicolon-delimited.
2. Parsed the `Date` and `Time` columns into a single `datetime` field using `parse_dates`.
3. Converted the power measurement columns to numeric values and treated invalid entries like `?` as missing values.
4. Sorted the dataset by datetime so it follows a clean chronological order.
5. Used `.resample('H').mean()` to aggregate the data by hour, which reduced the size from minute-by-minute records to hourly averages.
6. Split the datetime into separate `date` and `time` columns to make the dataset easier to read and filter.
7. Saved the reduced version to `reduced_data` for downstream analysis and visualisation.

This process makes the dataset smaller, cleaner, and more suitable for analysis while preserving the overall energy patterns.